# Mantenimiento Predictivo con NASA CMAPSS

Pipeline completo de **mantenimiento predictivo** usando el dataset CMAPSS (Commercial Modular
Aero-Propulsion System Simulation) de la NASA — subconjunto FD001.

| Sección | Técnica |
|---|---|
| 3 | Carga y preprocesamiento |
| 4 | Análisis con PCA |
| 5 | Series de tiempo (ARIMA) |
| 6 | Detección de anomalías (Isolation Forest) |
| 7 | Clasificación con Random Forest |
| 8 | Clasificación con LSTM |
| 9 | LSTM + Atención |
| 10 | Comparación por motor |

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score, f1_score
)

import statsmodels.api as sm

import tensorflow as tf
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import LSTM, Dense, Input, Layer
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

# Reproducibilidad
np.random.seed(42)
tf.random.set_seed(42)

# Rutas de datos
DATA_DIR = Path('mantenimiento_predictivo/data/CMaps')

print('TF version:', tf.__version__)
print('Data dir exists:', DATA_DIR.exists())

## 2. Funciones auxiliares
### 2.1 Helpers de visualización (Plotly)

In [ ]:
PLOTLY_TEMPLATE = 'plotly_white'


def plot_hist(data, title, xlabel, nbins=50):
    fig = px.histogram(
        x=data, nbins=nbins, opacity=0.75,
        title=title, labels={'x': xlabel},
        template=PLOTLY_TEMPLATE
    )
    fig.update_layout(showlegend=False)
    fig.show()


def plot_line(y, title, xlabel, ylabel, x=None):
    _x = x if x is not None else list(range(len(y)))
    fig = px.line(
        x=_x, y=y, title=title,
        labels={'x': xlabel, 'y': ylabel},
        template=PLOTLY_TEMPLATE
    )
    fig.show()


def plot_two_lines(x, y1, y2, labels, title, xlabel, ylabel,
                   styles=None, hline=None):
    dash1 = 'solid' if not styles else styles[0]
    dash2 = 'dash'  if not styles else styles[1]
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=x, y=y1, name=labels[0], line=dict(dash=dash1)
    ))
    fig.add_trace(go.Scatter(
        x=x, y=y2, name=labels[1], line=dict(dash=dash2)
    ))
    if hline is not None:
        fig.add_hline(
            y=hline, line_dash='dot', line_color='red',
            annotation_text='Umbral', annotation_position='bottom right'
        )
    fig.update_layout(
        title=title, xaxis_title=xlabel, yaxis_title=ylabel,
        template=PLOTLY_TEMPLATE
    )
    fig.show()

### 2.2 Construcción de secuencias, splits y métricas

In [ ]:
def make_sequences(df, sensor, window=30):
    """Ventanas deslizantes univariadas."""
    sequences, labels, units = [], [], []
    for unit_id, group in df.groupby('unit'):
        values  = group[sensor].values
        targets = group['distress'].values
        for i in range(len(values) - window):
            sequences.append(values[i:i + window])
            labels.append(targets[i + window])
            units.append(unit_id)
    return np.array(sequences), np.array(labels), np.array(units)


def make_lstm_sequences(df, sensor_cols, window=30):
    """Ventanas deslizantes multivariadas."""
    sequences, labels, units = [], [], []
    for unit_id, group in df.groupby('unit'):
        values  = group[sensor_cols].values
        targets = group['distress'].values
        for i in range(len(values) - window):
            sequences.append(values[i:i + window])
            labels.append(targets[i + window])
            units.append(unit_id)
    return np.array(sequences), np.array(labels), np.array(units)


def unit_time_series_splits(units: np.ndarray, n_splits: int = 5):
    """
    Divide cronológicamente por unidad (motor).
    Evita data leakage: test siempre contiene motores posteriores a train.
    """
    unique_units = np.array(sorted(pd.unique(units)))
    tscv = TimeSeriesSplit(n_splits=n_splits)
    for train_u_idx, test_u_idx in tscv.split(unique_units):
        train_units = set(unique_units[train_u_idx])
        test_units  = set(unique_units[test_u_idx])
        train_idx = np.where(np.isin(units, list(train_units)))[0]
        test_idx  = np.where(np.isin(units, list(test_units)))[0]
        yield train_idx, test_idx


def compute_metrics(y_true, y_prob):
    """Accuracy, ROC-AUC, PR-AUC y F1 para clasificación binaria."""
    y_hat = (y_prob >= 0.5).astype(int)
    return {
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'roc_auc':  float(roc_auc_score(y_true, y_prob))
                    if len(np.unique(y_true)) > 1 else np.nan,
        'pr_auc':   float(average_precision_score(y_true, y_prob))
                    if len(np.unique(y_true)) > 1 else np.nan,
        'f1':       float(f1_score(y_true, y_hat))
                    if len(np.unique(y_hat)) > 1 else np.nan,
    }

## 3. Carga y preprocesamiento

Columnas del dataset:
- `unit` — ID del motor
- `time` — ciclo operacional
- `op_setting_1/2/3` — condiciones de operación
- `sensor_1 … sensor_21` — lecturas de sensores

**Etiqueta**: `distress = True` cuando `RUL < 20` ciclos.

In [ ]:
COLS = (
    ['unit', 'time', 'op_setting_1', 'op_setting_2', 'op_setting_3']
    + [f'sensor_{i}' for i in range(1, 22)]
)

df = pd.read_csv(DATA_DIR / 'train_FD001.txt', sep=r'\s+', header=None)
df.dropna(axis=1, inplace=True)
df.columns = COLS

rul_gt = pd.read_csv(DATA_DIR / 'RUL_FD001.txt', header=None)
rul_gt.columns = ['max_RUL']
rul_gt['unit'] = rul_gt.index + 1

# RUL dentro del set de entrenamiento
last_cycle = df.groupby('unit')['time'].max().reset_index()
last_cycle.columns = ['unit', 'max_time']
df = df.merge(last_cycle, on='unit')
df['RUL']     = df['max_time'] - df['time']
df['distress'] = df['RUL'] < 20

print(f'Filas: {len(df):,}  |  Motores: {df["unit"].nunique()}')
print(f'Ciclos por motor — min: {last_cycle.max_time.min()}  '
      f'max: {last_cycle.max_time.max()}  '
      f'media: {last_cycle.max_time.mean():.1f}')
print(f'Proporción en distress: {df["distress"].mean():.2%}')
df.head()

In [ ]:
selected_sensors = ['sensor_9', 'sensor_14', 'sensor_4', 'sensor_3', 'sensor_17', 'sensor_2']

plot_hist(df['RUL'], title='Distribución de Vida Útil Restante (RUL)', xlabel='RUL (ciclos)')

In [ ]:
# Evolución de sensor_9 para los primeros 5 motores
fig = go.Figure()
for uid in range(1, 6):
    g = df[df['unit'] == uid]
    fig.add_trace(go.Scatter(
        x=g['time'], y=g['sensor_9'],
        mode='lines', name=f'Motor {uid}', opacity=0.8
    ))
fig.update_layout(
    title='Sensor 9 — evolución por motor',
    xaxis_title='Ciclo', yaxis_title='Sensor 9',
    template=PLOTLY_TEMPLATE
)
fig.show()

## 4. Análisis con PCA

Scaler y PCA se ajustan **solo sobre los primeros 30 ciclos** (operación saludable)
para que el espacio de referencia no esté contaminado por el deterioro.

In [ ]:
healthy_mask = df['time'] <= 30

scaler_vis = StandardScaler().fit(df.loc[healthy_mask, selected_sensors])
X_scaled_vis = scaler_vis.transform(df[selected_sensors])

pca_vis = PCA(n_components=3).fit(X_scaled_vis[healthy_mask.values])
df[['pca_1', 'pca_2', 'pca_3']] = pca_vis.transform(X_scaled_vis)

var_exp = pca_vis.explained_variance_ratio_
print('Varianza explicada por PC:', var_exp.round(3))
print(f'Acumulada PC1-PC3: {var_exp.cumsum()[-1]:.2%}')

In [ ]:
# Varianza explicada
fig = px.bar(
    x=['PC1', 'PC2', 'PC3'], y=var_exp,
    title='Varianza explicada por componente',
    labels={'x': 'Componente', 'y': 'Varianza explicada'},
    template=PLOTLY_TEMPLATE
)
fig.show()

In [ ]:
# Nube saludable en espacio PCA
healthy_df = df[df['time'] <= 30]

fig = px.scatter(
    healthy_df, x='pca_1', y='pca_2',
    title='PCA — Zona de operación saludable (primeros 30 ciclos)',
    opacity=0.4, template=PLOTLY_TEMPLATE
)
fig.show()

In [ ]:
# Distancia euclidiana al centroide saludable como indicador de salud
center = healthy_df[['pca_1', 'pca_2']].values.mean(axis=0)
df['pca_distance'] = np.linalg.norm(df[['pca_1', 'pca_2']].values - center, axis=1)

threshold_pca = df['pca_distance'].mean() + 3 * df['pca_distance'].std()
df['pca_anomaly'] = df['pca_distance'] > threshold_pca

print(f'Umbral PCA (μ + 3σ): {threshold_pca:.3f}')
print(f'Anomalías detectadas: {df["pca_anomaly"].sum():,} ({df["pca_anomaly"].mean():.2%})')

In [ ]:
# Distancia de salud a lo largo del tiempo para motor 1
m1 = df[df['unit'] == 1]
fig = go.Figure()
fig.add_trace(go.Scatter(x=m1['time'], y=m1['pca_distance'], mode='lines', name='Distancia PCA'))
fig.add_hline(y=threshold_pca, line_dash='dash', line_color='red',
              annotation_text='Umbral', annotation_position='bottom right')
fig.update_layout(
    title='Indicador de salud PCA — Motor 1',
    xaxis_title='Ciclo', yaxis_title='Distancia al centroide',
    template=PLOTLY_TEMPLATE
)
fig.show()

## 5. Modelado de series de tiempo — ARIMA

ARIMA(1,1,1) sobre `pca_1` del motor 1. Los residuos fuera de ±2σ
indican comportamiento inusual.

In [ ]:
unit_df = df[df['unit'] == 1].copy()

arima_model = sm.tsa.ARIMA(unit_df['pca_1'], order=(1, 1, 1))
fit = arima_model.fit()
forecast  = fit.predict(start=1, end=len(unit_df), dynamic=False)
residuals = unit_df['pca_1'].iloc[1:].values - forecast.iloc[1:].values

print(fit.summary())

In [ ]:
sigma_r = residuals.std()
cycles_r = unit_df['time'].iloc[1:].values

fig = go.Figure()
fig.add_trace(go.Scatter(x=cycles_r, y=residuals, mode='lines', name='Residuo'))
fig.add_hline(y= 2 * sigma_r, line_dash='dash', line_color='red',   annotation_text='+2σ')
fig.add_hline(y=-2 * sigma_r, line_dash='dash', line_color='red',   annotation_text='-2σ')
fig.add_hline(y=0,            line_dash='dot',  line_color='grey')
fig.update_layout(
    title='Residuos ARIMA sobre PC1 — Motor 1',
    xaxis_title='Ciclo', yaxis_title='Residuo',
    template=PLOTLY_TEMPLATE
)
fig.show()

## 6. Detección de anomalías — Isolation Forest

Algoritmo no supervisado que aísla observaciones anómalas con árboles aleatorios.
Aplicado sobre las 3 componentes PCA con `contamination=0.05`.

In [ ]:
pca_features = df[['pca_1', 'pca_2', 'pca_3']].values

clf = IsolationForest(contamination=0.05, random_state=42)
df['anomaly_iforest'] = clf.fit_predict(pca_features) == -1

print(f'Anomalías IF: {df["anomaly_iforest"].sum():,} ({df["anomaly_iforest"].mean():.2%})')

In [ ]:
fig = px.histogram(
    df, x='pca_distance', nbins=60, opacity=0.65,
    title='Distribución de distancia PCA (Isolation Forest)',
    labels={'pca_distance': 'Distancia al centroide'},
    template=PLOTLY_TEMPLATE
)
fig.add_vline(x=threshold_pca, line_dash='dash', line_color='red',
              annotation_text=f'Umbral {threshold_pca:.2f}',
              annotation_position='top right')
fig.show()

In [ ]:
# Scatter PC1 vs PC2 coloreado por anomalía
fig = px.scatter(
    df, x='pca_1', y='pca_2',
    color=df['anomaly_iforest'].map({True: 'Anomalía', False: 'Normal'}),
    color_discrete_map={'Normal': '#636EFA', 'Anomalía': '#EF553B'},
    opacity=0.3, title='Isolation Forest — Anomalías en espacio PCA',
    labels={'color': ''},
    template=PLOTLY_TEMPLATE
)
fig.show()

## 7. Clasificación de secuencias — Random Forest

Ventanas de 30 ciclos sobre `sensor_9` con **TimeSeriesSplit por unidad**
para respetar el orden temporal y evitar data leakage.

In [ ]:
X_seq, y_seq, units_rf = make_sequences(df, 'sensor_9', window=30)
print(f'Secuencias: {X_seq.shape}  |  Distress: {y_seq.sum():,} ({y_seq.mean():.2%})')

In [ ]:
rf_metrics = []

for fold, (train_idx, test_idx) in enumerate(
    unit_time_series_splits(units_rf, n_splits=5), start=1
):
    X_train_rf, X_test_rf = X_seq[train_idx], X_seq[test_idx]
    y_train_rf, y_test_rf = y_seq[train_idx], y_seq[test_idx]

    rf_clf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf_clf.fit(X_train_rf, y_train_rf)

    y_pred_rf = rf_clf.predict_proba(X_test_rf)[:, 1]
    m = compute_metrics(y_test_rf, y_pred_rf)
    rf_metrics.append(m)
    print(f'Fold {fold}: {m}')

rf_mean = {k: np.nanmean([m[k] for m in rf_metrics]) for k in rf_metrics[0]}
print(f'\nMedia RF: {rf_mean}')

In [ ]:
plot_hist(
    y_pred_rf,
    title='Random Forest — Probabilidades de distress (último fold)',
    xlabel='Probabilidad predicha'
)

## 8. Clasificación con LSTM

Red LSTM multivariada:
- **Entrada**: ventana 30 ciclos × 6 sensores → reducida a 3 PCs
- **Scaler + PCA**: ajustados solo en train de cada fold
- **Salida**: probabilidad de distress (sigmoide)

In [ ]:
X, y, units_lstm = make_lstm_sequences(df, selected_sensors, window=30)
n_features = len(selected_sensors)
print(f'Secuencias LSTM: {X.shape}')

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=0),
]

lstm_metrics = []

for fold, (train_idx, test_idx) in enumerate(
    unit_time_series_splits(units_lstm, n_splits=5), start=1
):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    # Scaler y PCA: ajustar SOLO en train
    scaler_lstm = StandardScaler().fit(X_train.reshape(-1, n_features))
    X_train_sc  = scaler_lstm.transform(X_train.reshape(-1, n_features)).reshape(X_train.shape)
    X_test_sc   = scaler_lstm.transform(X_test.reshape(-1, n_features)).reshape(X_test.shape)

    pca_lstm    = PCA(n_components=3).fit(X_train_sc.reshape(-1, n_features))
    X_train_seq = pca_lstm.transform(X_train_sc.reshape(-1, n_features)).reshape(X_train.shape[0], X_train.shape[1], 3)
    X_test_seq  = pca_lstm.transform(X_test_sc.reshape(-1, n_features)).reshape(X_test.shape[0], X_test.shape[1], 3)

    model_lstm = Sequential([
        LSTM(64, input_shape=(X_train_seq.shape[1], 3)),
        Dense(1, activation='sigmoid')
    ])
    model_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    model_lstm.fit(
        X_train_seq, y_train,
        epochs=20, batch_size=32, shuffle=False,
        validation_data=(X_test_seq, y_test),
        callbacks=callbacks, verbose=0
    )

    y_pred_lstm = model_lstm.predict(X_test_seq, verbose=0).flatten()
    m = compute_metrics(y_test, y_pred_lstm)
    lstm_metrics.append(m)
    print(f'Fold {fold}: {m}')

lstm_mean = {k: np.nanmean([m[k] for m in lstm_metrics]) for k in lstm_metrics[0]}
print(f'\nMedia LSTM: {lstm_mean}')

In [ ]:
plot_line(
    y_pred_lstm,
    title='LSTM — Probabilidades de distress (último fold)',
    xlabel='Índice de muestra', ylabel='Probabilidad predicha'
)

## 9. LSTM + Mecanismo de Atención

La capa de atención aprende a **ponderar los pasos de tiempo** más relevantes,
añadiendo interpretabilidad sobre qué parte de la ventana temporal importa más.

In [ ]:
class AttentionWithWeights(Layer):
    """Capa de atención con pesos exportables para visualización."""

    def build(self, input_shape):
        self.W = self.add_weight(
            name='attn_weight',
            shape=(input_shape[-1], 1),
            initializer='random_normal',
            trainable=True
        )
        super().build(input_shape)

    def call(self, inputs):
        scores  = tf.matmul(inputs, self.W)           # (batch, timesteps, 1)
        weights = tf.nn.softmax(scores, axis=1)       # normalización temporal
        context = tf.reduce_sum(inputs * weights, axis=1)  # representación comprimida
        return context, weights

In [ ]:
# Entrenamos sobre el último fold (scaler_lstm y pca_lstm del último fold disponibles)
input_seq = Input(shape=(X_train_seq.shape[1], 3))
lstm_out  = LSTM(64, return_sequences=True)(input_seq)
context, attn_weights = AttentionWithWeights()(lstm_out)
output    = Dense(1, activation='sigmoid', name='pred')(context)

model_attn = Model(inputs=input_seq, outputs={'pred': output, 'attn': attn_weights})
model_attn.compile(
    optimizer='adam',
    loss={'pred': 'binary_crossentropy'},
    metrics={'pred': 'accuracy'}
)
model_attn.summary()

In [ ]:
history = model_attn.fit(
    X_train_seq, {'pred': y_train},
    epochs=10, batch_size=32, shuffle=False,
    validation_data=(X_test_seq, {'pred': y_test}),
    callbacks=callbacks, verbose=1
)

In [ ]:
# Curva de entrenamiento
fig = go.Figure()
fig.add_trace(go.Scatter(y=history.history['pred_loss'],     mode='lines', name='Train loss'))
fig.add_trace(go.Scatter(y=history.history['val_pred_loss'], mode='lines', name='Val loss',  line=dict(dash='dash')))
fig.update_layout(
    title='LSTM + Atención — Curva de entrenamiento',
    xaxis_title='Época', yaxis_title='Loss (Binary Crossentropy)',
    template=PLOTLY_TEMPLATE
)
fig.show()

## 10. Comparación por motor

Comparamos las predicciones de LSTM y LSTM+Atención a lo largo
de toda la vida de un motor específico.

In [ ]:
UNIT_ID = 3
WINDOW  = 30

engine_data = df[df['unit'] == UNIT_ID].copy()

X_engine_raw = np.array([
    engine_data[selected_sensors].values[i:i + WINDOW]
    for i in range(len(engine_data) - WINDOW)
])
X_engine_sc = scaler_lstm.transform(
    X_engine_raw.reshape(-1, n_features)
).reshape(X_engine_raw.shape)
X_engine = pca_lstm.transform(
    X_engine_sc.reshape(-1, n_features)
).reshape(X_engine_raw.shape[0], WINDOW, 3)

cycles = engine_data['time'].values[WINDOW:]
print(f'Motor {UNIT_ID}: {len(cycles)} puntos de predicción')

In [ ]:
pred_lstm_engine = model_lstm.predict(X_engine, verbose=0).flatten()
pred_dict        = model_attn.predict(X_engine, verbose=0)
pred_attn_engine = pred_dict['pred'].flatten()

In [ ]:
plot_two_lines(
    x=cycles,
    y1=pred_lstm_engine,
    y2=pred_attn_engine,
    labels=['LSTM', 'LSTM + Atención'],
    title=f'Motor {UNIT_ID} — LSTM vs LSTM + Atención',
    xlabel='Ciclo', ylabel='Probabilidad de distress',
    styles=['solid', 'dash'],
    hline=0.5
)

In [ ]:
# Pesos de atención promedio para el motor (muestra qué ciclos de la ventana pesan más)
attn_map = pred_dict['attn'].numpy().squeeze(-1)  # (n_samples, timesteps)
avg_attn = attn_map.mean(axis=0)

fig = px.bar(
    x=list(range(1, WINDOW + 1)), y=avg_attn,
    title=f'Motor {UNIT_ID} — Pesos de atención promedio por paso de tiempo',
    labels={'x': 'Paso en la ventana', 'y': 'Peso de atención'},
    template=PLOTLY_TEMPLATE
)
fig.show()

## 11. Resumen de métricas

In [ ]:
summary = pd.DataFrame({
    'Modelo':   ['Random Forest', 'LSTM'],
    'Accuracy': [rf_mean['accuracy'],  lstm_mean['accuracy']],
    'ROC-AUC':  [rf_mean['roc_auc'],   lstm_mean['roc_auc']],
    'PR-AUC':   [rf_mean['pr_auc'],    lstm_mean['pr_auc']],
    'F1':       [rf_mean['f1'],        lstm_mean['f1']],
}).set_index('Modelo').round(4)

display(summary)

# Gráfica de barras comparativa
fig = px.bar(
    summary.reset_index().melt(id_vars='Modelo', var_name='Métrica', value_name='Valor'),
    x='Métrica', y='Valor', color='Modelo', barmode='group',
    title='Comparación de métricas — RF vs LSTM (media 5-fold)',
    template=PLOTLY_TEMPLATE
)
fig.update_yaxes(range=[0, 1])
fig.show()